In [1]:
import os
import cv2
import numpy as np
from ultralytics import YOLO

modelv12m = YOLO('Assets/yolov12m-face.pt')

In [2]:
# Download the dataset from 'https://www.kaggle.com/datasets/vishesh1412/celebrity-face-image-dataset'. Change the folder name to 'CelebrityFacesDataset' & place it inside 'Assets' folder.

In [3]:
folder_path = 'Assets/CelebrityFacesDataset'

folders = [
    folder
    for folder in os.listdir(folder_path)
    if os.path.isdir(os.path.join(folder_path, folder))
]

folders.sort()
folders

['Angelina Jolie',
 'Brad Pitt',
 'Denzel Washington',
 'Hugh Jackman',
 'Jennifer Lawrence',
 'Johnny Depp',
 'Kate Winslet',
 'Leonardo DiCaprio',
 'Megan Fox',
 'Natalie Portman',
 'Nicole Kidman',
 'Robert Downey Jr',
 'Sandra Bullock',
 'Scarlett Johansson',
 'Tom Cruise',
 'Tom Hanks',
 'Will Smith']

In [ ]:
# Download the face_recognizer.yml file for 'https://drive.google.com/file/d/1mlmVlVPH_Jelpeb5ijOXoT4Ph-9hNAJb/view?usp=drive_link' & place it inside the 'Assets' folder.

# Run this code if you don't want to train the model (Important: Download the dataset & run the above cells.).
'''
label_names = {}

for label, folder_name in enumerate(folders):
    folder = os.path.join(folder_path, folder_name)
    label_names[label] = folder_name
'''

In [13]:
faces = []
labels = []
label_names = {}

for label, folder_name in enumerate(folders):
    folder = os.path.join(folder_path, folder_name)
    label_names[label] = folder_name

    files = os.listdir(folder)

    for file in files:
        photo_path = os.path.join(folder, file)
        photo = cv2.imread(photo_path)

        if photo is None:
            continue

        face_result = modelv12m(photo, verbose = False)

        if len(face_result[0].boxes) == 0:
            continue

        left, top, right, bottom = face_result[0].boxes.xyxy[0].int()

        face = photo[top:bottom, left:right]
        face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
        face = cv2.resize(face, (250, 250))

        faces.append(face)
        labels.append(label)

In [16]:
face_recognizer = cv2.face.LBPHFaceRecognizer_create()

face_recognizer.train(faces, np.array(labels))
face_recognizer.write('Assets/face_recognizer.yml')

In [6]:
webcam = cv2.VideoCapture(0)

max_distance = 45
face_recognizer = cv2.face.LBPHFaceRecognizer_create()
face_recognizer.read('Assets/face_recognizer.yml')

while cv2.waitKey(1) & 0xFF != 27:
    ret, frame = webcam.read()

    if not ret:
        break

    face_result = modelv12m(frame, verbose = False)

    processed_feed = face_result[0].plot(
        labels = False,
        conf = False
    )

    for box in face_result[0].boxes.xyxy:
        left, top, right, bottom = box.int()

        face = frame[top:bottom, left:right]
        face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
        face = cv2.resize(face, (250, 250))

        pred_label, distance = face_recognizer.predict(face)

        if distance > max_distance:
            name = 'unknown'
        else:
            name = label_names[pred_label]
        label_height = 30

        cv2.rectangle(
            processed_feed,
            (int(left) - 2, int(bottom)),
            (int(right) + 2, int(bottom)+ label_height),
            (255, 0, 0),
            -1
        )

        cv2.putText(
            processed_feed,
            name + " " + str(int(distance)),
            (int(left) + 2, int(bottom) + 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 255, 255),
            2
        )

        cv2.imshow('Celebrity Face Recognition', processed_feed)

webcam.release()
cv2.destroyAllWindows()